In [1]:
from dotenv import load_dotenv
import os
from typing import List, Dict, Any, Optional, Union

load_dotenv()

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

# Import from our Classes module
from Classes.model_classes import SQLLineageExtractor
from Classes.helper_classes import SQLLineageResult
from Classes.regexp_extractor import RegexSQLExtractor
from Classes.validation_classes import SQLLineageValidator
from Classes.prompt_refiner import GigaChatBatchSQLLineageAgent


GIGACHAT_CREDENTIALS = os.environ.get("GIGACHAT_CREDENTIALS")

# Get the current working directory
current_dir = Path.cwd()

# Construct the path relative to current directory
file_path = current_dir / 'data' / 'SQL.txt'
file_path_valid = current_dir / 'data' / 'SQL_valid.txt'
file_path_logs = current_dir / 'data' / 'logs_batches.json'
wrong_sql_path = current_dir / 'data' / 'worng_sql_parsed.csv'


# Read file with example
with open(file_path, "r", encoding="utf-8") as f:
    SQL = f.read()

# Read file with example (valid for LLM)
with open(file_path_valid, "r", encoding="utf-8") as f:
    SQL_valid = f.read()

PROMPT = """You are a SQL lineage extraction expert. Extract source-to-target lineage from the SQL statement below and return **ONLY** a JSON object with two keys: "target" and "sources".

**SQL Statement:**
{sql_text}

**Extraction Rules:**
1. **Target**: The main object being created/modified (fully qualified name).
2. **Sources**: All base tables/views referenced in the query (fully qualified names, no aliases).
3. Strip all quotes (e.g., "schema"."table" → schema.table).
4. Exclude derived tables, CTEs, system tables.
5. Remove duplicates from sources.
6. Use lowercase for all names.

**Output Format:**
{format_instructions}

**Example:**
SQL: INSERT INTO analytics.sales_summary SELECT p.category, SUM(s.amount) FROM products.raw_data p JOIN sales.transactions s ON p.id = s.product_id;
Output: {{"target": "analytics.sales_summary", "sources": ["products.raw_data", "sales.transactions"]}}

**Your Response (JSON only):**"""

/Users/nikolajabramov/PycharmProjects/llm4s2t/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
SQLs = pd.read_csv(wrong_sql_path)['ddl'].to_list()

In [3]:
re_extractor = RegexSQLExtractor()
expected_results = [re_extractor.extract(sql) for sql in SQLs]

In [4]:
# Create extractor
extractor = SQLLineageExtractor(
    credentials=GIGACHAT_CREDENTIALS,   # or set env var GIGACHAT_API_KEY
    model="GigaChat-Pro",
    verify_ssl_certs=False,
    temperature=0.01
)

agent = GigaChatBatchSQLLineageAgent(
    credentials=GIGACHAT_CREDENTIALS,   # or set env var GIGACHAT_CREDENTIALS
    model="GigaChat-Pro",
    verify_ssl_certs=False,
    temperature=0.01, 
    max_concurrent=1, 
    extractor = extractor
)

In [5]:
result = await agent.optimize_prompt_batch(SQLs, expected_results=expected_results, verbose=True, output_file=file_path_logs)


STARTING BATCH PROMPT OPTIMIZATION (GigaChat Agent)

📝 Initial prompt preview:
You are a SQL lineage extraction expert. Extract source-to-target lineage from the SQL statement below and return **ONLY** a JSON object with two keys: "target" and "sources".

**SQL Statement:**
{sql...


################################################################################
ITERATION 1 / 5
################################################################################

################################################################################
VALIDATE BATCH NODE - Iteration 1/5
################################################################################

BATCH REFLECTION - Iteration 1
✓ Improved prompt generated (1264 chars)

################################################################################
VALIDATE BATCH NODE - Iteration 2/5
################################################################################
⚠️ Extraction failed after 3 attempts: Extractor returned error: 

In [6]:
print(result["optimized_prompt"])

You are a precise SQL lineage extraction assistant. Your task is to extract source-to-target lineage from the given SQL statement and produce **EXACTLY ONE JSON OBJECT**, adhering to the following strict rules:

- Use precisely two keys: `"target"` (a string) and `"sources"` (an array of strings).
- For `"target"`:
  * Identify the main table or view being created or updated.
  * Specify its fully qualified name as `schema.table` (without quotes), converted to lowercase.
- For `"sources"`:
  * Include only direct base tables or views referenced in the query (ignore derived tables, CTEs, temp tables, or system tables).
  * Format each source as a fully qualified name `schema.table`, unquoted, and in lowercase.
  * Remove any duplicate entries.
- Ensure your response matches the exact JSON format defined by `{format_instructions}`, including proper syntax and structure.
- Return **ONLY** the requested JSON object, containing no additional text or comments.

Input SQL: {sql_text}

Output 

In [7]:
# Construct the path relative to current directory
file_path = current_dir / 'data' / 'views.csv'

#Create a validaition Class
validation = SQLLineageValidator()


In [8]:
df_data = pd.read_csv(file_path)
df_data['ddl'] = "INSERT INTO s_grnplm_vd_t_bvd_db_dmslcl." + df_data['table_name'] + " " + df_data['view_def']

In [10]:
REFINED_PROMPT = result["optimized_prompt"]

# Create extractor with refined prompt
extractor_refined = SQLLineageExtractor(
    credentials=GIGACHAT_CREDENTIALS,   # or set env var GIGACHAT_API_KEY
    model="GigaChat-Pro",
    verify_ssl_certs=False,
    temperature=0.01,
    template=REFINED_PROMPT
)

In [11]:
data_lineage = []
f1_scores = {}
results = []

for index, row in tqdm(df_data.iterrows(), total=len(df_data), desc="🎨 Extracting S2T"):

    target_ = re_extractor.extract(row['ddl'])

    res_ = await validation.run_comprehensive_validation(
        extractor_refined, 
        row['ddl'],
        expected_result=target_)
    results.append(res_)
    try:
        f1_scores[row['table_name']] = res_['metrics']['f1_score']
    except:
        f1_scores[row['table_name']] = 0
        
    data_lineage.append(res_['result'])

🎨 Extracting S2T:   0%|          | 0/127 [00:00<?, ?it/s]

In [12]:
np.mean(list(f1_scores.values()))

np.float64(0.9335602111471965)

In [13]:
df = pd.DataFrame(results)
df.pivot_table(index='status', columns='validation_type', values='message', aggfunc='count').fillna(0.0)

validation_type,comprehensive,derived_tables,sources,target,uniqueness
status,,,,,
FAILED,0.0,1.0,2.0,1.0,2.0
SUCCESS,121.0,0.0,0.0,0.0,0.0


In [14]:
df_success = df[df['status'] == 'SUCCESS'].join(pd.DataFrame(
    df[df['status'] == 'SUCCESS'].pop('metrics').tolist(),
    index=df[df['status'] == 'SUCCESS'].index,
    columns=['precision', 'recall', 'f1_score']
))

df_success['f1_score'].mean()

np.float64(0.9798524530222641)